# # Film Eleştirileri ve Bag-of-Words Modellemesi

🎯 Bu zorluğun amacı, metinlerin ***Bag-of-words*** modellemesiyle oynamaktır.

✍️ Aşağıdaki veri setinde, _“olumlu”_ veya _“olumsuz”_ olarak sınıflandırılmış 2000 adet yorum bulunmaktadır.

In [3]:
import pandas as pd
import urllib.request
import ssl
import io

# SSL kontrolünü devre dışı bırakan özel bir bağlam (context) oluşturuyoruz
context = ssl._create_unverified_context()

# URL'den veriyi bu bağlamı kullanarak manuel olarak çekiyoruz
url = "https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv"
with urllib.request.urlopen(url, context=context) as response:
    csv_verisi = response.read()

# İndirdiğimiz veriyi Pandas'ın anlayacağı formata (BytesIO) sokup yüklüyoruz
data = pd.read_csv(io.BytesIO(csv_verisi))

# Bakalım olmuş mu?
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [4]:
data.shape

(2000, 2)

## 1. Ön işleme (Preprocessing)

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek bir `preprocessing` fonksiyonu yazın ve bunu tüm yorumlara uygulayın. Fonksiyon şunları yapmalıdır:
    - boşlukları kaldırma
    - harfleri küçük harfe çevirme
    - sayıları kaldırma
    - noktalama işaretlerini kaldırma
    - tokenization (kelimelere ayırma)
    - lemmatization (kelime köküne indirgeme)
- Temizlenmiş yorumları `clean_reviews` adlı bir sütunda saklayabilirsiniz.
- Bu aşamada stopword’leri kaldırmayın; nedenini `3. N-gram modelleme` bölümünde açıklayacağız.

In [5]:
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocessing(sentence):
   import re

# Lemmatizer objesini dışarıda tanımlayalım ki her seferinde yeniden oluşmasın
lemmatizer = WordNetLemmatizer()

def preprocessing(sentence):
    # 1. Başındaki ve sonundaki boşlukları kaldırma
    sentence = sentence.strip()
    
    # 2. Harfleri küçük harfe çevirme
    sentence = sentence.lower()
    
    # 3. Sayıları kaldırma (regex kullanarak)
    sentence = re.sub(r'\d+', '', sentence)
    
    # 4. Noktalama işaretlerini kaldırma
    sentence = "".join([char for char in sentence if char not in string.punctuation])
    
    # 5. Tokenization (Kelimelere ayırma)
    tokens = word_tokenize(sentence)
    
    # 6. Lemmatization (Kelime köküne indirgeme)
    lem_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Çıktının tek bir dize (string) olması için birleştiriyoruz
    return " ".join(lem_tokens)


In [10]:
import re
import string
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# 1. Gerekli objeyi tanımlayalım
lemmatizer = WordNetLemmatizer()

# 2. Preprocessing fonksiyonunu hatasız tanımlayalım
def preprocessing(sentence):
    # Boşluk temizliği ve küçük harf
    sentence = str(sentence).strip().lower()
    
    # Sayıları kaldırma (İşte 're' burada lazım!)
    sentence = re.sub(r'\d+', '', sentence)
    
    # Noktalama işaretlerini kaldırma
    sentence = "".join([char for char in sentence if char not in string.punctuation])
    
    # Kelimelere ayırma ve köke inme (Lemmatization)
    tokens = word_tokenize(sentence)
    lem_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(lem_tokens)

# 3. Fonksiyonu doğru sütun adıyla (reviews) uygulayalım
data['clean_reviews'] = data['reviews'].apply(preprocessing)

# 4. Hedef değişkeni (target) sayısallaştıralım
le = LabelEncoder()
data['target_encoded'] = le.fit_transform(data['target'])

# Sonucu görelim
print("İşlem tamam! İşte ilk 5 satır:")
data[['target', 'target_encoded', 'clean_reviews']].head()

İşlem tamam! İşte ilk 5 satır:


,target,target_encoded,clean_reviews
0,neg,0,plot two teen couple go to a church party drin...
1,neg,0,the happy bastard quick movie review damn that...
2,neg,0,it is movie like these that make a jaded movie...
3,neg,0,quest for camelot is warner bros first feature...
4,neg,0,synopsis a mentally unstable man undergoing ps...


❓ **Soru (LabelEncoding)**❓

Hedefinizi LabelEncode ile kodlayın ve `“target_encoded”` adlı bir sütuna kaydedin.

In [11]:
from sklearn.preprocessing import LabelEncoder

# Hedef değişkeni (target) sayısallaştırma (pos -> 1, neg -> 0)
le = LabelEncoder()
data['target_encoded'] = le.fit_transform(data['target'])

# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


In [12]:
# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


## 2. Bag-of-Words Modellemesi

❓ **Soru (Tek kelimelik sözcüklerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

# 1. Metinleri tekli kelimeler (unigram) olarak vektörleştiriyoruz
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(data['clean_reviews'])
y = data['target_encoded']

# 2. Multinomial Naive Bayes modelini tanımlıyoruz
naive_bayes = MultinomialNB()

# 3. cross_validate ile modelin başarısını puanlıyoruz (5 katlı çapraz doğrulama)
cv_sonuclari = cross_validate(naive_bayes, X_bow, y, cv=5, scoring='accuracy')

# 4. Ortalama skoru yazdırıyoruz
ortalama_skor = cv_sonuclari['test_score'].mean()
print(f"Tek kelimelik (Unigram) Naive Bayes Ortalama Doğruluğu: % {ortalama_skor * 100:.2f}")

Tek kelimelik (Unigram) Naive Bayes Ortalama Doğruluğu: % 81.65


## 3. N-gram Modellemesi

👀 Stop kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini bigramlarla eğiteceğiz. Bu nedenle, “I do not like coriander” (Kişnişi sevmiyorum) gibi bir cümlede, örneğin bu cümlede olumsuzluğu tespit etmek için “do not” bigramını taramak önemlidir.

❓ **Soru (bigramlarla NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [14]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

🏁 Tebrikler! Artık vektörleştirilmiş metinler üzerinde Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

